<a href="https://colab.research.google.com/github/iffathsaleem/RFM-Market-Analysis-Stock-Segmentation./blob/main/Market_Analysis_and_Stock_Segmentation_Using_RFM_CW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Task 1 Data Preparation and Feature Engineering**

## 1.1 Ticker selection and data acquisition

In [36]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [37]:
start = '2021-01-01'
end = '2025-01-01'
benchmark_ticker = '^GSPC'

In [38]:
from pathlib import Path

In [39]:
from google.colab import drive
drive.mount('/content/drive')

project_folder = Path('/content/drive/MyDrive/Colab Notebooks/CRM Coursework 1')
output_folder = project_folder / 'CRM_CM_Enhanced_Results'
data_folder = output_folder / 'raw_data'
data_folder.mkdir(parents=True, exist_ok=True)
print('Save location:', output_folder)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save location: /content/drive/MyDrive/Colab Notebooks/CRM Coursework 1/CRM_CM_Enhanced_Results


In [40]:
USE_SAVED_DATA = (data_folder / 'SP500_Companies.csv').exists()

In [41]:
print('Reuse saved data:', USE_SAVED_DATA)

Reuse saved data: True


In [42]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
ticker_file = data_folder/ 'SP500_Companies.csv'

In [43]:
if ticker_file.exists():
    sp500_df = pd.read_csv(ticker_file)
    print('Using the saved constituent list. Check ticker_source.csv for its date.')
else:
    tables = pd.read_html(url, storage_options={'User-Agent': 'Mozilla/5.0'})
    sp500_df = tables[0]
    pass
    sp500_df['Ticker'] = sp500_df['Symbol'].str.replace('.', '-', regex=False)
    sp500_df.to_csv(ticker_file, index=False)

    ticker_source = pd.DataFrame([{
        'Source': url,
        'RetrievedUTC': pd.Timestamp.now(tz='UTC').isoformat(),
        'Start': start,
        'EndExclusive': end
    }])
    ticker_source.to_csv(data_folder / 'ticker_source.csv', index=False)

Using the saved constituent list. Check ticker_source.csv for its date.


In [44]:
tickers = sp500_df['Ticker'].tolist()
print('Total tickers:', len(tickers))
sp500_df[['Ticker', 'Security', 'GICS Sector']].head(10)

Total tickers: 503


,Ticker,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology
5,ADBE,Adobe Inc.,Information Technology
6,AMD,Advanced Micro Devices,Information Technology
7,AES,AES Corporation,Utilities
8,AFL,Aflac,Financials
9,A,Agilent Technologies,Health Care


In [45]:
import time

In [46]:
def download_data(ticker):
    filename = ticker.replace('^', 'index_') + '.csv'
    saved_file = data_folder / filename

    if saved_file.exists():
        data = pd.read_csv(saved_file, index_col=0, parse_dates=True)
    else:
        if USE_SAVED_DATA:
            raise ValueError('No cached history; refer to original acquisition failures')
        data = pd.DataFrame()
        for attempt in range(3):
            try:
                data = yf.Ticker(ticker).history(
                    start=start, end=end, auto_adjust=False, actions=True
                )
                if not data.empty:
                    break
            except Exception as error:
                print(ticker, error)
            time.sleep(2)

        if data.empty:
            raise ValueError('Download failed after three attempts')

        data.index = pd.to_datetime(data.index).tz_localize(None).normalize()
        data.to_csv(saved_file)

        download_record = pd.DataFrame([{
            'Ticker': ticker,
            'RetrievedUTC': pd.Timestamp.now(tz='UTC').isoformat(),
            'Start': start,
            'EndExclusive': end,
            'AutoAdjust': False
        }])
        download_record.to_csv(
            data_folder / (ticker.replace('^', 'index_') + '_source.csv'),
            index=False
        )

    data.index = pd.to_datetime(data.index).tz_localize(None).normalize()
    return data.sort_index()

In [47]:
market_data = download_data(benchmark_ticker)
market_data.head()

,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2021-01-04,3764.610107,3769.989990,3662.709961,3700.649902,3700.649902,5015000000,0.0,0.0
2021-01-05,3698.020020,3737.830078,3695.070068,3726.860107,3726.860107,4591020000,0.0,0.0
2021-01-06,3712.199951,3783.040039,3705.340088,3748.139893,3748.139893,6064110000,0.0,0.0
2021-01-07,3764.709961,3811.550049,3764.709961,3803.790039,3803.790039,5099160000,0.0,0.0
2021-01-08,3815.050049,3826.689941,3783.600098,3824.679932,3824.679932,4773040000,0.0,0.0


In [48]:
downloaded_data = {}
download_errors = []


In [49]:
for position, ticker in enumerate(tickers, start=1):
    try:
        downloaded_data[ticker] = download_data(ticker)
    except Exception as error:
        download_errors.append({'Ticker': ticker, 'Reason': str(error)})

    if position % 25 == 0:
        print('Processed', position, 'of', len(tickers))

Processed 25 of 503
Processed 50 of 503
Processed 75 of 503
Processed 100 of 503
Processed 125 of 503
Processed 150 of 503
Processed 175 of 503
Processed 200 of 503
Processed 225 of 503
Processed 250 of 503
Processed 275 of 503
Processed 300 of 503
Processed 325 of 503
Processed 350 of 503
Processed 375 of 503
Processed 400 of 503
Processed 425 of 503
Processed 450 of 503
Processed 475 of 503
Processed 500 of 503


In [50]:
pd.DataFrame(download_errors, columns=['Ticker', 'Reason']).to_csv(
    output_folder / 'download_errors.csv', index=False
)

In [51]:
print('Shares downloaded:', len(downloaded_data))
print('Acquisition failures:', len(download_errors))

Shares downloaded: 498
Acquisition failures: 5
